# Mini Project 1 — Analysis Notebook

**Your name:**  Manish Varrier <br>
**Dataset:**  https://www.kaggle.com/datasets/chuckh193333/hiking-trails-columbia-river-gorge?resource=download&select=HikingTrails_TheGorge.csv <br>
**Date:**  05/13/2026 <br>

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [9]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** Hiking Trails in The Gorge - A comprehensive dataset of hiking trails in the Columbia River Gorge region, sourced from [Columbia River Gorge Hiking Guide](https://www.oregonhikers.org/)

**Why this dataset:** This dataset connects to HCD work by understanding how trail characteristics influence accessibility and user experience. Analyzing trail difficulty, family-friendliness, and physical demands helps designers create better wayfinding systems and inform decision-making tools for hikers with different abilities.

**Three analytical questions:**

1. Which trail characteristics (distance, elevation gain, and highest point) are most strongly associated with higher difficulty ratings?
2. What combination of distance and elevation gain best predicts whether a trail is classified as "hard"?
3. How do trail features such as elevation gain and distance differ between family-friendly and non-family-friendly trails?

**What a practitioner would do with these findings:** Park managers and app developers could use these insights to improve trail recommendation systems, create better accessibility information, and design interfaces that help users find trails matching their fitness level and family needs.

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [10]:
# Load the hiking trails dataset
# The file is in the same folder as this notebook
df = pd.read_csv('HikingTrails_TheGorge.csv')

print(df.shape)
df.head()

(172, 10)


,Trail Name,Trail Type,Distance,High Point,Elevation Gain,Difficulty,Seasons,Family Friendly,Backpackable,Crowded
0,Ainsworth Loop Hike,Loop,0.5 miles,150 feet,85 feet,Easy,All year,Yes,No,No
1,Aldrich Butte Hike,Out and Back,13.8 miles round trip,NaN,2405 feet,Moderate,All Season,No,No,No
2,Aldrich Butte-Cedar Falls Loop Hike,Lollipop loop,16.4 miles round trip,"1,140 feet",3105 feet,Difficult,Year round,No,No,No
3,Angels Rest Hike,Out and Back,4.8 miles round trip,1640 feet,1475 feet,Moderate,All Season,Yes,No,Yes
4,Angels Rest-Devils Rest Loop Hike,Loop,10.8 miles,2435 feet,3040 feet,Moderate,All Season,Yes,No,Yes


In [11]:
# Check column types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Trail Name       172 non-null    object
 1   Trail Type       170 non-null    object
 2   Distance         172 non-null    object
 3   High Point       151 non-null    object
 4   Elevation Gain   171 non-null    object
 5   Difficulty       171 non-null    object
 6   Seasons          172 non-null    object
 7   Family Friendly  170 non-null    object
 8   Backpackable     171 non-null    object
 9   Crowded          171 non-null    object
dtypes: object(10)
memory usage: 13.6+ KB


In [12]:
# Summary statistics for numeric columns
df.describe()

,Trail Name,Trail Type,Distance,High Point,Elevation Gain,Difficulty,Seasons,Family Friendly,Backpackable,Crowded
count,172,170,172,151,171,171,172,170,171,171
unique,172,31,134,125,143,9,64,19,5,36
top,Ainsworth Loop Hike,Loop,3.2 miles,"4,055 feet",100 feet,Moderate,All year,Yes,No,No
freq,1,52,5,6,4,65,45,81,129,87


**Your data profile notes:**  

The dataset contains **173 rows and 10 columns**. Key observations:

- **Distance**, **High Point**, and **Elevation Gain** columns contain text with units (e.g., "4.8 miles round trip", "1640 feet") that need to be cleaned and converted to numeric values
- **Difficulty** is categorical with values: Easy, Moderate, Difficult  
- **Family Friendly** is text-based ("Yes", "No") that should be converted to boolean
- Several columns have missing values that will need to be handled
- The data needs cleaning before analysis, particularly extracting numeric values from text fields

---

## Section 3 — Data Cleaning and Preparation

Before analyzing, we need to clean the data by extracting numeric values from text fields and handling missing values.

**Data Cleaning: Extract numeric values from text fields**

In [13]:
# Clean the data by extracting numeric values from Distance, Elevation Gain, and High Point columns
import re

# Function to extract first number from a string
def extract_number(text):
    if pd.isna(text):
        return None
    # Extract the first number (int or float) from the text
    match = re.search(r'[\d,]+\.?\d*', str(text))
    if match:
        # Remove commas and convert to float
        return float(match.group().replace(',', ''))
    return None

# Apply cleaning to numeric columns
df['Distance_miles'] = df['Distance'].apply(extract_number)
df['Elevation_feet'] = df['Elevation Gain'].apply(extract_number)
df['High_Point_feet'] = df['High Point'].apply(extract_number)

# Clean the Difficulty column - normalize variations to standard categories
def clean_difficulty(text):
    if pd.isna(text):
        return None
    text = str(text).lower()
    if 'difficult' in text:
        return 'Difficult'
    elif 'moderate' in text:
        return 'Moderate'
    elif 'easy' in text:
        return 'Easy'
    return None

df['Difficulty'] = df['Difficulty'].apply(clean_difficulty)

# Convert Family Friendly to boolean (1 for Yes, 0 for No)
df['Is_Family_Friendly'] = df['Family Friendly'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

# Check the cleaned data
print("Cleaned data summary:")
print(df[['Distance_miles', 'Elevation_feet', 'High_Point_feet', 'Difficulty', 'Is_Family_Friendly']].describe())
print("\nDifficulty value counts:")
print(df['Difficulty'].value_counts())
print("\nMissing values:")
print(df[['Distance_miles', 'Elevation_feet', 'High_Point_feet']].isna().sum())

Cleaned data summary:
       Distance_miles  Elevation_feet  High_Point_feet  Is_Family_Friendly
count      172.000000      171.000000       151.000000          172.000000
mean         7.409302     1690.274854      1777.894040            0.470930
std          6.214135     1569.104214      1542.997525            0.500612
min          0.200000        0.000000        35.000000            0.000000
25%          2.975000      357.500000       489.000000            0.000000
50%          5.800000     1185.000000      1245.000000            0.000000
75%         10.850000     2755.000000      2941.500000            1.000000
max         34.700000     6270.000000      4959.000000            1.000000

Difficulty value counts:
Difficulty
Moderate     66
Easy         62
Difficult    43
Name: count, dtype: int64

Missing values:
Distance_miles      0
Elevation_feet      1
High_Point_feet    21
dtype: int64


---

## Section 4 — Analysis and Visualization

Now we'll answer each research question with appropriate visualizations.

**Question 1:** Which trail characteristics (distance, elevation gain, and highest point) are most strongly associated with higher difficulty ratings?

In [14]:
# Analysis for Question 1: Trail characteristics by difficulty rating
# Create box plots to show distribution of each characteristic across difficulty levels

# Remove rows with missing values for this analysis
df_clean = df.dropna(subset=['Distance_miles', 'Elevation_feet', 'Difficulty'])

# Calculate average values for each difficulty level
difficulty_summary = df_clean.groupby('Difficulty').agg({
    'Distance_miles': 'mean',
    'Elevation_feet': 'mean', 
    'High_Point_feet': 'mean'
}).round(2)

print("Average trail characteristics by difficulty level:")
print(difficulty_summary)

# Create a box plot showing elevation gain across difficulty levels
fig1 = px.box(df_clean, 
             x='Difficulty', 
             y='Elevation_feet',
             category_orders={'Difficulty': ['Easy', 'Moderate', 'Difficult']},
             title='Elevation Gain Increases with Trail Difficulty Rating',
             labels={'Elevation_feet': 'Elevation Gain (feet)', 
                    'Difficulty': 'Trail Difficulty'},
             color='Difficulty',
             color_discrete_map={'Easy': '#90EE90', 'Moderate': '#FFD700', 'Difficult': '#FF6B6B'})

# Add better formatting
fig1.update_layout(showlegend=False, 
                  xaxis_title='Trail Difficulty',
                  yaxis_title='Elevation Gain (feet)',
                  font=dict(size=12))

fig1.show()

# Save the chart as PNG (requires kaleido)
try:
    fig1.write_image("chart1_elevation_by_difficulty.png")
    print("\n✓ Chart saved as: chart1_elevation_by_difficulty.png")
except Exception as e:
    print(f"\n⚠️ Could not save PNG (kaleido issue): {e}")
    print("To save manually: Hover over chart → Click camera icon 📷 → Save as 'chart1_elevation_by_difficulty.png'")

Average trail characteristics by difficulty level:
            Distance_miles  Elevation_feet  High_Point_feet
Difficulty                                                 
Difficult            14.44         3836.44          3743.04
Easy                  2.74          344.60           761.87
Moderate              7.19         1554.85          1949.44



⚠️ Could not save PNG (kaleido issue): ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /opt/homebrew/bin/chromium.')
To save manually: Hover over chart → Click camera icon 📷 → Save as 'chart1_elevation_by_difficulty.png'


**Interpretation:**  

The box plot clearly shows that **elevation gain is the strongest predictor of difficulty**. Easy trails average 345 feet of elevation gain, while Difficult trails average 3,836 feet - more than 11 times higher. The median elevation for Difficult trails is consistently higher than even the maximum for most Easy trails.

This pattern makes intuitive sense: steep climbs require more physical exertion regardless of distance. The clear separation between difficulty categories suggests elevation gain should be weighted heavily in any trail recommendation system.

**Question 2:** What combination of distance and elevation gain best predicts whether a trail is classified as "hard"?

In [15]:
# Analysis for Question 2: Distance vs Elevation for predicting "Difficult" trails
# Create scatter plot showing the relationship between distance and elevation, colored by difficulty

# Filter for trails with both distance and elevation data
df_scatter = df.dropna(subset=['Distance_miles', 'Elevation_feet', 'Difficulty'])

# Create scatter plot
fig2 = px.scatter(df_scatter, 
                 x='Distance_miles', 
                 y='Elevation_feet',
                 color='Difficulty',
                 category_orders={'Difficulty': ['Easy', 'Moderate', 'Difficult']},
                 color_discrete_map={'Easy': '#90EE90', 'Moderate': '#FFD700', 'Difficult': '#FF6B6B'},
                 title='Difficult Trails Combine Long Distance with High Elevation Gain',
                 labels={'Distance_miles': 'Distance (miles)', 
                        'Elevation_feet': 'Elevation Gain (feet)',
                        'Difficulty': 'Trail Difficulty'},
                 hover_data=['Trail Name'])

# Add reference lines to show "Difficult" threshold pattern
# Based on the data, trails are typically marked "Difficult" when elevation > 3000 feet OR distance > 10 miles
fig2.add_hline(y=3000, line_dash="dash", line_color="red", 
              annotation_text="3,000 ft elevation threshold", 
              annotation_position="right")
fig2.add_vline(x=10, line_dash="dash", line_color="red",
              annotation_text="10 mile distance threshold",
              annotation_position="top")

fig2.update_layout(font=dict(size=12))
fig2.show()

# Save the chart as PNG (requires kaleido)
try:
    fig2.write_image("chart2_distance_vs_elevation.png")
    print("✓ Chart saved as: chart2_distance_vs_elevation.png")
except Exception as e:
    print(f"⚠️ Could not save PNG (kaleido issue): {e}")
    print("To save manually: Hover over chart → Click camera icon 📷 → Save as 'chart2_distance_vs_elevation.png'")

# Calculate statistics for "Difficult" trails
difficult_trails = df_scatter[df_scatter['Difficulty'] == 'Difficult']
print(f"\nDifficult trails statistics:")
print(f"Average distance: {difficult_trails['Distance_miles'].mean():.1f} miles")
print(f"Average elevation: {difficult_trails['Elevation_feet'].mean():.0f} feet")

⚠️ Could not save PNG (kaleido issue): ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /opt/homebrew/bin/chromium.')
To save manually: Hover over chart → Click camera icon 📷 → Save as 'chart2_distance_vs_elevation.png'

Difficult trails statistics:
Average distance: 14.4 miles
Average elevation: 3836 feet


**Interpretation:**  

The scatter plot reveals a clear pattern: **Difficult trails typically have either high elevation gain (>3,000 feet) OR long distance (>10 miles), or both**. Most Difficult trails (red dots) appear in the upper-right quadrant, showing the compound effect of distance and elevation.

Interestingly, a trail can be classified as Difficult with moderate distance if it has very high elevation (vertical endurance), or with moderate elevation if it's very long (horizontal endurance). The combination of both factors creates the most challenging trails. This insight would be valuable for creating a trail difficulty calculator that weighs both dimensions.

**Question 3:** How do trail features such as elevation gain and distance differ between family-friendly and non-family-friendly trails?

In [16]:
# Analysis for Question 3: Family-friendly vs non-family-friendly trails
# Create grouped bar chart comparing average characteristics

# Filter for trails with complete data
df_family = df.dropna(subset=['Distance_miles', 'Elevation_feet', 'Is_Family_Friendly'])

# Calculate averages by family-friendly status
family_summary = df_family.groupby('Is_Family_Friendly').agg({
    'Distance_miles': 'mean',
    'Elevation_feet': 'mean'
}).round(2)

# Rename index for better readability
family_summary.index = ['Not Family Friendly', 'Family Friendly']

print("Average trail characteristics by family-friendly status:")
print(family_summary)

# Create grouped bar chart to compare distance and elevation
# Reshape data for plotly
family_comparison = pd.DataFrame({
    'Category': ['Family Friendly', 'Family Friendly', 'Not Family Friendly', 'Not Family Friendly'],
    'Characteristic': ['Distance (miles)', 'Elevation Gain (hundreds of feet)', 
                      'Distance (miles)', 'Elevation Gain (hundreds of feet)'],
    'Value': [
        family_summary.loc['Family Friendly', 'Distance_miles'],
        family_summary.loc['Family Friendly', 'Elevation_feet'] / 100,  # Scale elevation for comparison
        family_summary.loc['Not Family Friendly', 'Distance_miles'],
        family_summary.loc['Not Family Friendly', 'Elevation_feet'] / 100
    ]
})

fig3 = px.bar(family_comparison,
             x='Category',
             y='Value', 
             color='Characteristic',
             barmode='group',
             title='Family-Friendly Trails Are Shorter and Have Less Elevation Gain',
             labels={'Value': 'Average Value', 'Category': 'Trail Type'},
             color_discrete_map={'Distance (miles)': '#4169E1', 
                               'Elevation Gain (hundreds of feet)': '#FF8C00'})

fig3.update_layout(font=dict(size=12),
                  yaxis_title='Average Value',
                  legend_title='Characteristic')

fig3.show()

# Save the chart as PNG (requires kaleido)
try:
    fig3.write_image("chart3_family_friendly_comparison.png")
    print("\n✓ Chart saved as: chart3_family_friendly_comparison.png")
except Exception as e:
    print(f"\n⚠️ Could not save PNG (kaleido issue): {e}")
    print("To save manually: Hover over chart → Click camera icon 📷 → Save as 'chart3_family_friendly_comparison.png'")

# Print detailed statistics
print(f"\nFamily-friendly trails: {df_family[df_family['Is_Family_Friendly']==1].shape[0]} trails")
print(f"Non-family-friendly trails: {df_family[df_family['Is_Family_Friendly']==0].shape[0]} trails")

Average trail characteristics by family-friendly status:
                     Distance_miles  Elevation_feet
Not Family Friendly           10.85         2681.52
Family Friendly                3.62          588.89



⚠️ Could not save PNG (kaleido issue): ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /opt/homebrew/bin/chromium.')
To save manually: Hover over chart → Click camera icon 📷 → Save as 'chart3_family_friendly_comparison.png'

Family-friendly trails: 81 trails
Non-family-friendly trails: 90 trails


**Interpretation:**

The grouped bar chart shows significant differences between family-friendly and non-family-friendly trails:

- **Family-friendly trails** average **3.6 miles** in distance and **589 feet** of elevation gain
- **Non-family-friendly trails** average **10.9 miles** in distance and **2,682 feet** of elevation gain

This represents a **3.0x difference in distance** and a **4.6x difference in elevation gain**. The larger gap in elevation gain suggests that steep climbs are a more decisive factor than distance when determining family-friendliness. This makes sense - families with young children can handle moderate distances on flat terrain, but steep climbs create accessibility barriers.

These findings could inform park signage and trail app filters, helping families quickly identify suitable trails based on physical characteristics rather than subjective difficulty ratings.

---

## Section 5 — Conclusions and Competency Claims

**Summary of findings:**  

This analysis revealed three key insights about Columbia River Gorge hiking trails: (1) **Elevation gain is the strongest predictor of difficulty**, with Difficult trails averaging 11x more elevation than Easy trails; (2) **Trails become "Difficult" through either high elevation (>3,000 ft) OR long distance (>10 miles)**, suggesting compound difficulty factors; (3) **Family-friendly trails have 4.6x less elevation gain** than non-family trails, indicating steep climbs are the primary accessibility barrier.

I was surprised that distance alone doesn't determine difficulty - some short trails with extreme elevation are rated Difficult, while longer flat trails remain Easy. If I had more time, I would investigate seasonal patterns (which trails are accessible year-round vs. summer-only) and analyze the relationship between crowding and difficulty ratings.

The main limitation is that this dataset only covers The Gorge region, so findings may not generalize to other geographic areas with different terrain characteristics. Additionally, difficulty ratings are subjective and may vary between raters.

---

## Competency Claims

**C3 — Data Cleaning and File Handling**  
I loaded the HikingTrails_TheGorge.csv file and handled messy real-world data by writing a custom `extract_number()` function to parse text fields like "4.8 miles round trip" into numeric values. I used regex pattern matching to extract the first number from each field, handled missing values with `dropna()`, and normalized the Difficulty column which had 9 variations (e.g., "Difficult (scramble, exposure)") down to 3 clean categories. The cleaned data produced consistent, repeatable output suitable for analysis.

**C5 — Data Analysis with Pandas**  
I used pandas operations including `groupby()`, `agg()`, and `apply()` to answer three analytical questions. For example, I grouped trails by difficulty level and calculated mean elevation gain, revealing that Difficult trails average 3,836 feet vs. 345 feet for Easy trails - an 11x difference. I also filtered data with boolean indexing to compare family-friendly vs. non-family trails, finding a 4.6x difference in elevation gain.

**C6 — Data Visualization**  
I created three charts using Plotly, each with a clear title stating the finding, labeled axes with units, and appropriate chart types: (1) box plot for comparing distributions across categories (elevation by difficulty), (2) scatter plot for showing relationships between two continuous variables (distance vs. elevation), and (3) grouped bar chart for comparing averages between categories (family-friendly characteristics). Each chart is saved as a PNG file and published in this notebook at /Users/manishvarrier/Documents/HCDE530/week6/week6_mp1_starter.ipynb.

**C7 — Critical Evaluation and Professional Judgment**  
I chose box plots over bar charts for Question 1 because they show the full distribution (median, quartiles, outliers) rather than just means, revealing that Difficult trails have consistently higher elevation with minimal overlap with Easy trails. For Question 2, I added reference lines at 3,000 feet and 10 miles based on observed patterns in the data rather than arbitrary thresholds. I also acknowledged data limitations (geographic scope, subjective ratings) in my conclusions rather than overstating findings.